# Tabula Sapiens: IMSTE and UMAP embeddings

Fit UMAP and IMSTE once on a balanced Tabula Sapiens sample, then show the same three layouts with two independent colorings: tissue and cell type. The sample cap is applied to one selected annotation (`SAMPLE_BY`) and the resulting cells are shared by both rows.

The expression workflow starts from the raw-count layer, normalizes each cell to 10,000 counts, applies `log1p`, selects highly variable genes, scales them, and computes PCA. Labels are used only for sampling and coloring, never as embedding features.

In [ ]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import time
import shutil
import zipfile
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "imste" / "__init__.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository from the notebook working directory.")
for local_path in (PROJECT_ROOT, PROJECT_ROOT / "notebooks" / "src"):
    local_path = str(local_path)
    if local_path in sys.path:
        sys.path.remove(local_path)
    sys.path.insert(0, local_path)
import imste
if not Path(imste.__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()):
    raise RuntimeError("This kernel loaded IMSTE from another location; restart and run from the top.")
from imste import IteratedMinimumSpanningTreeEmbedder

DATA_DIR = PROJECT_ROOT / "data" / "tabula_sapiens"
ARCHIVE_PATH = DATA_DIR / "TabulaSapiens_v1.h5ad.zip"
H5AD_PATH = DATA_DIR / "TabulaSapiens.h5ad"
if not H5AD_PATH.is_file():
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(f"Expected the dataset archive at {ARCHIVE_PATH}.")
    try:
        with zipfile.ZipFile(ARCHIVE_PATH) as archive:
            h5ad_members = [name for name in archive.namelist() if name.lower().endswith(".h5ad")]
            if len(h5ad_members) != 1:
                raise RuntimeError(f"Expected one H5AD file in {ARCHIVE_PATH.name}; found {h5ad_members}.")
            member = archive.getinfo(h5ad_members[0])
            required_bytes = member.file_size + 1_000_000_000
            if shutil.disk_usage(DATA_DIR).free < required_bytes:
                raise OSError(
                    f"Not enough free space to extract {member.file_size / 1e9:.1f} GB; "
                    f"need at least {required_bytes / 1e9:.1f} GB free."
                )
            print(f"Extracting {member.filename} ({member.file_size / 1e9:.1f} GB)...")
            with archive.open(member) as source, H5AD_PATH.open("wb") as destination:
                shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
    except zipfile.BadZipFile as error:
        raise RuntimeError(
            "The Tabula Sapiens ZIP is incomplete or invalid. Wait for the download to finish, then rerun this cell."
        ) from error
    print(f"Extracted dataset to {H5AD_PATH}; the source ZIP is retained.")

RANDOM_STATE = 42
MAX_CELLS_PER_CLASS = 300  # Set to None to use every cell.
SAMPLE_BY = "cell_type"  # "cell_type" or "tissue"; both rows use this same sample.
TARGET_SUM = 10_000
N_HVG = 2_000
N_PCS = 50
METRIC = 'euclidean'

## Load annotations and cap cells per class

The source object is opened in backed mode so the full expression matrix is not loaded just to inspect labels. We sample up to `MAX_CELLS_PER_CLASS` cells within each class of `SAMPLE_BY`; setting it to `None` disables the cap. A fixed random seed makes the selected cells reproducible.

In [ ]:
adata_source = sc.read_h5ad(H5AD_PATH, backed="r")

# Tabula Sapiens v1 annotation names; prefer the atlas' ontology cell types.
def first_available(columns, candidates, description):
    for name in candidates:
        if name in columns:
            return name
    raise KeyError(f"Could not find {description}; available obs columns: {list(columns)}")

tissue_column = first_available(
    adata_source.obs.columns, ("organ_tissue", "tissue", "tissue_general"), "tissue annotation"
)
cell_type_column = first_available(
    adata_source.obs.columns, ("cell_ontology_class", "cell_type", "free_annotation"), "cell-type annotation"
)
if SAMPLE_BY not in {"cell_type", "tissue"}:
    raise ValueError("SAMPLE_BY must be 'cell_type' or 'tissue'.")
sampling_column = cell_type_column if SAMPLE_BY == "cell_type" else tissue_column

obs = adata_source.obs[[tissue_column, cell_type_column]].copy()
obs = obs.fillna({tissue_column: "unannotated", cell_type_column: "unannotated"}).astype(str)
if MAX_CELLS_PER_CLASS is not None:
    if not isinstance(MAX_CELLS_PER_CLASS, int) or MAX_CELLS_PER_CLASS < 1:
        raise ValueError("MAX_CELLS_PER_CLASS must be a positive integer or None.")
    rng = np.random.default_rng(RANDOM_STATE)
    selected_groups = []
    group_values = obs[sampling_column].to_numpy()
    for class_name in pd.unique(group_values):
        members = np.flatnonzero(group_values == class_name)
        selected_groups.append(rng.choice(members, min(len(members), MAX_CELLS_PER_CLASS), replace=False))
    selected_indices = np.sort(np.concatenate(selected_groups))
else:
    selected_indices = np.arange(adata_source.n_obs)

print(f"Source: {adata_source.n_obs:,} cells × {adata_source.n_vars:,} genes")
print(f"Selected: {len(selected_indices):,} cells; capped by {SAMPLE_BY!r} at {MAX_CELLS_PER_CLASS}")
print(f"Tissue annotation: {tissue_column}; cell-type annotation: {cell_type_column}")
display(obs.iloc[selected_indices].groupby([tissue_column, cell_type_column], observed=True).size()
        .rename("cells").reset_index().head(20))

## Normalize counts and compute the shared feature matrix

Use the raw count layer, normalize each cell to `TARGET_SUM`, and apply `log1p`. Then select highly variable genes, scale, and reduce to `N_PCS` principal components. These PCs are the shared high-dimensional input for both UMAP and IMSTE.

In [ ]:
# Tabula Sapiens v1 stores unnormalized counts in layers['raw_counts'].
if "raw_counts" not in adata_source.layers:
    raise KeyError(f"Expected layers['raw_counts']; available layers: {list(adata_source.layers.keys())}")

adata_sample = adata_source[selected_indices].to_memory()
adata_source.file.close()
adata = sc.AnnData(
    X=adata_sample.layers["raw_counts"].copy(),
    obs=obs.iloc[selected_indices].copy(),
    var=adata_sample.var.copy(),
)
del adata_sample

# Standard single-cell normalization: equalize total counts per cell, then log transform.
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat")
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=min(N_PCS, adata.n_obs - 1, adata.n_vars - 1), random_state=RANDOM_STATE)
X = np.asarray(adata.obsm["X_pca"], dtype=np.float32)
tissue_labels = adata.obs[tissue_column].astype(str).to_numpy()
cell_type_labels = adata.obs[cell_type_column].astype(str).to_numpy()
print(f"Feature matrix: {X.shape[0]:,} cells × {X.shape[1]} PCs from {adata.n_vars:,} HVGs")
display(pd.DataFrame({"cells": [len(X)], "tissues": [pd.Series(tissue_labels).nunique()],
                     "cell types": [pd.Series(cell_type_labels).nunique()]}))

## Fit the layouts once

UMAP and IMSTE are each fitted once. The plots below reuse these coordinate arrays for both annotation rows.

In [ ]:
import umap

UMAP_SETTINGS = {
    "n_components": 2,
    "n_neighbors": 15,
    "min_dist": 0.1,
    "metric": METRIC,
    "init": "recursive",
    "recursive_coarsening_ratio": 4,
    "compatibility_layout": False,
    "nearest_neighbor_backend": "rinnd",
    "random_state": RANDOM_STATE,
    "n_epochs": 200,
    "verbose": True,
}

IMSTE_SETTINGS = {
    "n_components": 2,
    "n_msts": 3,
    "n_epochs": 200,
    "mst_method": "famst",
    "mst_neighbors": 15,
    "metric": METRIC,
    "device": "auto",
    "random_state": RANDOM_STATE,
    "multilevel_init": True,
    "multilevel_reduction_ratio": 4,
    "multilevel_base_size": 1_024,
    "multilevel_epochs": 10,
    "gravity_mode": "density_core",
    "gravity_neighbors": 15,
    "gravity_resolution": 0.7,
    "gravity_strength": 0.3,
    "mean_shift_neighbors": 100,
    "mean_shift_bandwidth_factor": 7.0,
    "mean_shift_step_size": 0.9,
    "mean_shift_iterations": 30,
    "verbose": True,
}

t0 = time.perf_counter()
Z_umap = umap.UMAP(**UMAP_SETTINGS).fit_transform(X)
umap_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
imste_estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_SETTINGS)
Z_imste_base, Z_imste_refined = imste_estimator.fit_transform(X, return_intermediate=True)
imste_seconds = time.perf_counter() - t0
print(f"UMAP: {umap_seconds / 60:.1f} min; IMSTE including gravity and mean shift: {imste_seconds / 60:.1f} min")

## Compare the same layouts by tissue and cell type

Each row changes only the colors. Columns reuse UMAP, the initial IMSTE layout, and the refined IMSTE layout (gravity followed by mean shift). No layout is recomputed for the second annotation.

In [ ]:
from imste.visualization import render_datashader_embedding


def color_key_for(values):
    categories = list(pd.unique(values))
    cmap = plt.get_cmap("turbo", max(len(categories), 2))
    return {name: to_hex(cmap(i)) for i, name in enumerate(categories)}

layout_specs = [
    ("UMAP", Z_umap),
    ("IMSTE base", Z_imste_base),
    ("IMSTE gravity → mean shift", Z_imste_refined),
]
annotation_specs = [
    ("Tissue", tissue_labels),
    ("Cell type", cell_type_labels),
]

fig, axes = plt.subplots(2, 3, figsize=(21, 12), constrained_layout=True)
for row, (annotation_name, values) in enumerate(annotation_specs):
    color_key = color_key_for(values)
    for col, (layout_name, coordinates) in enumerate(layout_specs):
        image = render_datashader_embedding(
            coordinates[:, :2], values,
            width=1100, height=850,
            color_key=color_key,
            min_alpha=20,
            background="white",
            spread_px=1,
        )
        axes[row, col].imshow(image.to_pil())
        axes[row, col].set_title(f"{layout_name} · colored by {annotation_name.lower()}")
        axes[row, col].set_axis_off()

plt.show()

### Notes

- `MAX_CELLS_PER_CLASS` and `SAMPLE_BY` control the one shared sample. Choose `SAMPLE_BY = "tissue"` to cap by tissue instead of cell type, or set the cap to `None` to use all cells.
- The tissue and cell-type rows use identical cell indices and identical fitted coordinates.
- The source is the [Tabula Sapiens release 1.0](https://figshare.com/articles/dataset/Tabula_Sapiens_release_1_0/14267219). Its data-object documentation describes the raw count layer and annotations used here.